<a href="https://colab.research.google.com/github/chaseshallow/ansci_4040_final_class_project/blob/main/Step02bDetectTrackFull.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 02b · Detect and track cows: full hour

Runs YOLO26n (COCO "cow") + ByteTrack on the **whole** `ReferenceClip.mp4` (1 h) at 5 fps, with a 4x preview. Takes ~20 min on a T4. Run Step 02a first if you changed settings.

Writes into the Shared Drive, `CalvingPenPipelineOutputs/<video>/Step02bFull/`, **overwriting** each run:

- `Tracks.csv`: one row per box per processed frame (Frame, TimeSeconds, TrackId, X1..Y2, Confidence)
- `TrackSummary.csv`: one row per track
- `PreviewTracks.mp4`: the clip with boxes and TrackIds drawn; watch this to judge the result

A TrackId is not an AnimalId; that comes in later steps. Videos stay on Drive, never in git.
Needs a GPU: **Runtime > Change runtime type > T4 GPU**. The folder is shared: one person runs a step at a time. Edit cell 2 and rerun; don't add cells.

In [ ]:
# 1. Setup: mount Drive, get the pipeline code from GitHub, install YOLO.
from google.colab import drive
drive.mount("/content/drive")

import os, sys
RepoUrl = "https://github.com/chaseshallow/ansci_4040_final_class_project.git"
RepoBranch = "main"
RepoDir = "/content/ansci_4040_final_class_project"

if os.path.isdir(RepoDir):
    !git -C "{RepoDir}" fetch -q origin "{RepoBranch}" && git -C "{RepoDir}" checkout -q "{RepoBranch}" && git -C "{RepoDir}" pull -q --ff-only
else:
    !git clone -q --branch "{RepoBranch}" "{RepoUrl}" "{RepoDir}"
%pip install -q -r "{RepoDir}/CalvingPen/requirements.txt"
sys.path.insert(0, f"{RepoDir}/CalvingPen")

# Only needed after an update: Runtime > Restart session, then rerun from here.
from CalvingPipeline.DetectTrack import DetectAndTrack, EstimateRealFps
from CalvingPipeline.Paths import OutputRoot
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU")

In [ ]:
# 2. Settings: the only cell to edit.
VideoStem = "2026-09-26 00-00-00 Calving Pen 2"
VideoOutputDir = OutputRoot / VideoStem   # Shared Drive: CalvingPenPipelineOutputs/<video>/   # step 01 output
ClipPath = VideoOutputDir / "ReferenceClip.mp4"
OutputDir = VideoOutputDir / "Step02bFull"

TargetFps = 5          # frames per second processed; the cow moves slowly, 5 is plenty
Confidence = 0.25      # lower = more boxes (and more false ones)
MaxSeconds = None      # None = the whole clip
PreviewSeconds = None  # preview of the whole clip
PreviewSpeed = 4       # preview plays 4x faster (1 h -> ~15 min)

assert ClipPath.exists(), f"Run step 01 first: {ClipPath}"

In [ ]:
# 3. Run: detect + track, then show the summary and the preview video.
import pandas as pd
from IPython.display import Video, display

print(EstimateRealFps(ClipPath))   # share of repeated frames (high = the camera updates slowly)
Stats = DetectAndTrack(ClipPath, OutputDir, TargetFps=TargetFps, Confidence=Confidence, MaxSeconds=MaxSeconds,
                       PreviewSeconds=PreviewSeconds, PreviewSpeed=PreviewSpeed)

display(pd.read_csv(OutputDir / "TrackSummary.csv"))
PreviewPath = OutputDir / "PreviewTracks.mp4"
PreviewMb = PreviewPath.stat().st_size / 2**20
print(f"Preview: {PreviewPath} ({PreviewMb:.0f} MB)")
if PreviewMb < 50:   # bigger files are too heavy to play inside the notebook; open them from Drive
    display(Video(str(PreviewPath), embed=True, width=960))